In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": 1234,
    "database": "banking_project"
}

cfg = DB_CONFIG
url = f"mysql+pymysql://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['database']}"
engine = create_engine(url)

# First create empty table
create_table = """
CREATE TABLE IF NOT EXISTS dim_payment (
    customer_id BIGINT,
    loan_id BIGINT,
    installment_version FLOAT,
    installment_number FLOAT,
    scheduled_day FLOAT,
    actual_payment_day FLOAT,
    scheduled_amount FLOAT,
    actual_payment_amount FLOAT
)
"""

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS dim_payment"))
    conn.execute(text(create_table))
    conn.commit()
    print("Table created.")

# Load in chunks from staging
print("Loading data in chunks...")
chunk_size = 100000
offset = 0
total = 0

with engine.connect() as conn:
    while True:
        chunk_query = f"""
            SELECT
                sk_id_curr AS customer_id,
                sk_id_prev AS loan_id,
                num_instalment_version AS installment_version,
                num_instalment_number AS installment_number,
                days_instalment AS scheduled_day,
                days_entry_payment AS actual_payment_day,
                amt_instalment AS scheduled_amount,
                amt_payment AS actual_payment_amount
            FROM stg_installments
            LIMIT {chunk_size} OFFSET {offset}
        """
        chunk = pd.read_sql(chunk_query, conn)
        
        if len(chunk) == 0:
            break
        
        chunk.to_sql('dim_payment', engine, if_exists='append', index=False)
        total += len(chunk)
        offset += chunk_size
        print(f"  Inserted {total} rows...")

print(f"Done. Total rows: {total}")

Table created.
Loading data in chunks...
  Inserted 100000 rows...
  Inserted 200000 rows...
  Inserted 300000 rows...
  Inserted 400000 rows...
  Inserted 500000 rows...
  Inserted 600000 rows...
  Inserted 700000 rows...
  Inserted 800000 rows...
  Inserted 900000 rows...
  Inserted 1000000 rows...
  Inserted 1100000 rows...
  Inserted 1200000 rows...
  Inserted 1300000 rows...
  Inserted 1400000 rows...
  Inserted 1500000 rows...
  Inserted 1600000 rows...
  Inserted 1700000 rows...
